In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm

from abc import ABC, abstractmethod
from typing import Tuple, List, Optional

In [2]:
class BaseXANESAugmentation(ABC):
    '''
    Подразумевается модификация только спектра
    '''
    def __init__(self, p: float = 0.5):
        '''
        p - вероятность применения аугментации.
        '''
        assert 0.0 <= p <= 1.0, 'Вероятность p должна быть в диапазоне [0, 1]'
        self.p = p

    def __call__(self, spectrum: np.ndarray, energy:np.ndarray = None) -> np.ndarray:
        if np.random.rand() > self.p:
            return spectrum.copy()
        
        return self.apply(spectrum, energy)

    @abstractmethod
    def apply(self, spectrum: np.ndarray, energy: np.ndarray = None) -> np.ndarray:
        '''
        Реализация аугментации
        '''
        pass

class EnergyShift(BaseXANESAugmentation):
    '''
    Сдвиг края поглащения
    (Смещения шкалы E0 из-за калибровки монохроматора)
    '''
    def __init__(self, shift_range_ev: Tuple[float, float] = (-2.0, 2.0), p: float = 0.5):
        super().__init__(p=p)
        self.shift_range = shift_range_ev

    def apply(self, spectrum: np.ndarray, energy:np.ndarray) -> np.ndarray:
        shift = np.random.uniform(self.shift_range[0], self.shift_range[1])
        shifted_energy = energy - shift
        
        return np.interp(energy, shifted_energy, spectrum)


class EnergyScale(BaseXANESAugmentation):
    '''
    Равномерное растяжение/сжатие по энергии (Scale).
    (Деформация решетки / изотопный эффект без изменения координации)
    '''
    def __init__(self, scale_range: Tuple[float, float] = (0.97, 1.03), p: float = 0.5):
        super().__init__(p=p)
        self.scale_range = scale_range

    def apply(self, spectrum: np.ndarray, energy: np.ndarray) -> np.ndarray:
        scale = np.random.uniform(self.scale_range[0], self.scale_range[1])
        scaled_energy = energy * scale
        
        return np.interp(energy, scaled_energy, spectrum)


class GaussianBroadening(BaseXANESAugmentation):
    '''
    Гауссово уширение.
    Моделирует приборное разрешение и тепловой фактор Дебая–Валлера.
    '''
    def __init__(self, sigma_ev_range: Tuple[float, float] = (0.3, 0.8), p: float = 0.5):
        super().__init__(p=p)
        self.sigma_range = sigma_ev_range

    def apply(self, spectrum: np.ndarray, energy: np.ndarray) -> np.ndarray:
        sigma_ev = np.random.uniform(self.sigma_range[0], self.sigma_range[1])

        dE = np.abs(energy[1] - energy[0])
        sigma_pts = sigma_ev / dE

        radius = int(np.ceil(3 * sigma_pts))
        x = np.arange(-radius, radius + 1)
        
        kernel = np.exp(-0.5 * (x / sigma_pts) ** 2)
        kernel /= kernel.sum()  # Сохраняем нормировку/площадь
        
        # Свертка с отражением границ для предотвращения краевых эффектов
        padded_spectrum = np.pad(spectrum, radius, mode='reflect')
        conv_result = np.convolve(padded_spectrum, kernel, mode='valid')
        
        return conv_result


class XANESAugmentationPipeline:
    '''
    Последовательно прогоняет 1D спектр через список трансформаций.
    '''
    def __init__(self, transforms: List[BaseXANESAugmentation]):
        self.transforms = transforms

    def __call__(self, spectrum: np.ndarray, energy: np.ndarray) -> np.ndarray:
        augmented = spectrum.copy()
        for t in self.transforms:
            augmented = t(augmented, energy)
        return augmented

In [3]:
def augment_xanes_dataset(X_data: np.ndarray,
                          y_data: np.ndarray,
                          energy: np.ndarray,
                          pipeline: XANESAugmentationPipeline,
                          multiplier: float = 5.0,
                          keep_original: bool = True,
                          seed: Optional[int] = None
                         ) -> Tuple[np.ndarray, np.ndarray]:
    '''
    Оркестратор для аугментации данных.

    X_data - Массив спектров формы (N_samples, N_energy_points).
    y_data - Массив векторов CN формы (N_samples, N_spheres).
    multiplier - Коэффициент "умножения" данных (например, 3.0 означает, 
                       что итоговый размер составит 3 * N_samples).
    energy - 1D сетка энергий.
    pipeline - Объект XANESAugmentationPipeline.
    keep_original - Если True, оригинальные данные обязательно включаются в итоговый набор.
    seed - Seed для воспроизводимости.
    
    return - (X_augmented, y_augmented) — готовые NumPy массивы для сохранения или передачу в sklearn/CatBoost/PyTorch.
    '''
    if seed is not None:
        np.random.seed(seed)

    if multiplier < 1.0:
        raise ValueError("multiplier должен быть >= 1.0")

    num_samples = len(X_data)
    target_total = int(np.round(num_samples * multiplier))
    num_to_generate = target_total - num_samples if keep_original else target_total

    if num_to_generate <= 0:
        return X_data.copy(), y_data.copy()

    full_passes = num_to_generate // num_samples
    remainder = num_to_generate % num_samples

    generated_indices = []
    for _ in range(full_passes):
        generated_indices.extend(np.random.permutation(num_samples))
    if remainder > 0:
        generated_indices.extend(np.random.permutation(num_samples)[:remainder])

    X_generated = []
    y_generated = []

    for idx in generated_indices:
        spec_orig = X_data[idx]
        cn_target = y_data[idx]

        spec_aug = pipeline(spec_orig, energy=energy)

        X_generated.append(spec_aug)
        y_generated.append(cn_target.copy())

    X_gen_arr = np.array(X_generated)
    y_gen_arr = np.array(y_generated)

    if keep_original:
        X_out = np.vstack([X_data, X_gen_arr])
        y_out = np.vstack([y_data, y_gen_arr])
    else:
        X_out = X_gen_arr
        y_out = y_gen_arr

    shuffle_idx = np.random.permutation(len(X_out))
    return X_out[shuffle_idx], y_out[shuffle_idx]

In [4]:
X_train = np.load('../processed_data/DataSets/Combined_No_Aug_3A_Train_X.npz')['arr_0']
y_train = np.load('../processed_data/DataSets/Combined_No_Aug_3A_Train_y.npz')['arr_0']
X_test = np.load('../processed_data/DataSets/Combined_No_Aug_3A_Test_X.npz')['arr_0']
y_test = np.load('../processed_data/DataSets/Combined_No_Aug_3A_Test_y.npz')['arr_0']

In [5]:
unique_subs_df = pd.read_pickle('../source_data/INIT_unique_substructures_Rcut_7.5_stephist_0.1.pkl')

In [6]:
e_array = np.arange(-10,60.1, 0.5)

In [7]:
%%time
path_to_spectra = list(Path('../processed_data/conv_dense_spectra/').glob('**/*.txt'))
path_to_spectra = sorted(path_to_spectra,key=lambda x: int(x.name.split('.')[0]))

spectra = []
for path in tqdm(path_to_spectra):
    t_s = np.loadtxt(path)
    spectrum = np.interp(e_array, t_s[:,0], t_s[:,1])
    spectra.append(spectrum)


X_add_val = np.array(spectra)
y_add_val = np.array(unique_subs_df.CN.values.tolist())

  0%|          | 0/2151 [00:00<?, ?it/s]

CPU times: total: 6.83 s
Wall time: 6.83 s


In [8]:
pipeline = XANESAugmentationPipeline([EnergyShift(shift_range_ev=(-1.5, 1.5), p=0.5),
                                      EnergyScale(scale_range=(0.97, 1.03), p=0.5), 
                                      GaussianBroadening(sigma_ev_range=(0.3, 0.8), p=0.5)
                                     ])

In [9]:
%%time

X_train_new, y_train_new = augment_xanes_dataset(X_data = X_train,
                                                 y_data = y_train,
                                                 energy = e_array,
                                                 pipeline = pipeline,
                                                 multiplier = 5,
                                                 keep_original = True,
                                                 seed = 42
                                                )

X_test_new, y_test_new = augment_xanes_dataset(X_data = X_test,
                                               y_data = y_test,
                                               energy = e_array,
                                               pipeline = pipeline,
                                               multiplier = 5,
                                               keep_original = True,
                                               seed = 42
                                              )

X_add_val_new, y_add_val_new = augment_xanes_dataset(X_data = X_add_val,
                                                     y_data = y_add_val,
                                                     energy = e_array,
                                                     pipeline = pipeline,
                                                     multiplier = 5,
                                                     keep_original = True,
                                                     seed = 42
                                                    )

CPU times: total: 7.98 s
Wall time: 8 s


In [10]:
SAVE_DATA_PATH = Path('../processed_data/DataSets/')
np.savez(Path(SAVE_DATA_PATH, 'Scenario_3_Train_X'), X_train_new) 
np.savez(Path(SAVE_DATA_PATH, 'Scenario_3_Train_y'), y_train_new)
np.savez(Path(SAVE_DATA_PATH, 'Scenario_3_Test_X'), X_test_new)
np.savez(Path(SAVE_DATA_PATH, 'Scenario_3_Test_y'), y_test_new) 
np.savez(Path(SAVE_DATA_PATH, 'Scenario_3_Val_X'), X_add_val_new)
np.savez(Path(SAVE_DATA_PATH, 'Scenario_3_Val_y'), y_add_val_new) 

In [11]:
# plt.plot(e_array,X_add_val_new[0],'b-')
# plt.plot(e_array,X_add_val[0],'r:')
# print(y_add_val_new[0])
# print(y_add_val[0])